# 01 - Extracción MAIAC AOD desde NASA Earthdata (modo local)

**Producto**: MCD19A2 v061 (Optical_Depth_055, AOD_Uncertainty, Column_WV, AOD_QA).

**Tile MODIS para BBox Cali**: `h10v08` (NO `h09v07`).

**Periodo**: 2020-01-01 a 2024-12-31 (1.827 días, CMR reporta 1.822 granules disponibles).

**Volumen estimado**: 6,7 MB por granule, cerca de 2,4 GB/año en h10v08, total cerca de 12 GB en 5 años.

**Modo de ejecución**: local Windows/Linux con upload streaming a ADLS por granule (descargar -> hash -> subir -> borrar local). El output de tqdm/rich de `earthaccess.download` se redirige a archivo de log para evitar saturar el frontend del notebook.

**Destino**: `abfss://geovision@stanaliticafinal.dfs.core.windows.net/01-bronze/maiac/year=YYYY/month=MM/<granule>.hdf`

**Estado al 2026-05-04**: el run previo `20260504T011528Z-75f904dd` subió 150 granules (todos los DOY 1-31, equivalente a enero de los 5 años) y reportó 1.672 failed con `day is out of range for month`. Causa: bug en `parse_yyyy_mm` que pasaba `doy` directo como `day` a `datetime(year, 1, doy)`. Corregido en este notebook con `timedelta(days=doy - 1)`. Los 150 archivos en bucket fueron validados (size + SHA256 + header HDF-EOS2 OK contra manifest); este run reanuda los 1.672 faltantes vía idempotencia `fs.exists`.

**Snippet canónico**: `agent-docs/investigaciones/01-05-2026-bloqueantes/02-acceso-earthdata-maiac.md:148-174`.

In [1]:
import os
import sys
import time
import json
import hashlib
import secrets
import logging
import contextlib
from datetime import datetime, timezone, timedelta
from pathlib import Path

import yaml
import earthaccess
import adlfs
from azure.identity import DefaultAzureCredential

logging.getLogger("earthaccess").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

c:\Users\mitgar14\Desktop\Ingenieria de Datos e IA - UAO\Semestre 7\Analítica\Semana #12 - #17\Proyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Carga de .env local (cwd o repo root). Modo local-only, sin path Fabric.
from dotenv import load_dotenv

if load_dotenv(override=False):
    print("OK: .env cargado desde cwd local")
else:
    candidates = [Path.cwd() / ".." / ".." / ".env", Path.cwd() / ".." / ".env"]
    for c in candidates:
        c = c.resolve()
        if c.exists() and load_dotenv(c, override=False):
            print(f"OK: .env cargado desde {c}")
            break
    else:
        print("AVISO: .env no encontrado. Las credenciales deben estar ya en os.environ.")

for k in ("EARTHDATA_USER", "EARTHDATA_PASS"):
    assert os.environ.get(k), f"{k} faltante. Configurar en .env o exportar al entorno."

OK: .env cargado desde cwd local


In [3]:
# Configuración canónica del proyecto
BBOX_CALI = (-76.60, 3.30, -76.40, 3.55)  # lon_min, lat_min, lon_max, lat_max
PERIODO = ("2020-01-01", "2024-12-31")
TILE_MODIS = "h10v08"

# Storage canónico
STORAGE_ACCOUNT = "stanaliticafinal"
FILESYSTEM = "geovision"
BRONZE_PREFIX = "01-bronze/maiac"
META_PREFIX = "_meta/manifest/maiac"
LOG_PREFIX = "_meta/logs/maiac"

# Run id (sortable por timestamp + suffix random corto)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + secrets.token_hex(4)

# Path local persistente fuera del repo (override con env MAIAC_LOCAL_DIR)
LOCAL_BASE = Path(os.environ.get("MAIAC_LOCAL_DIR", str(Path.home() / "geovision-tmp" / "maiac")))
TMP_DIR = LOCAL_BASE / RUN_ID
TMP_DIR.mkdir(parents=True, exist_ok=True)

# Log file local (luego se sube a ADLS al final del run)
LOG_FILE = TMP_DIR / f"{RUN_ID}.log"
LOG_FILE.touch()

print(f"[{RUN_ID}] BBox: {BBOX_CALI}")
print(f"[{RUN_ID}] Periodo: {PERIODO}")
print(f"[{RUN_ID}] Tile: {TILE_MODIS}")
print(f"[{RUN_ID}] tmp local: {TMP_DIR}")
print(f"[{RUN_ID}] log: {LOG_FILE}")
print(f"[{RUN_ID}] destino: abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{BRONZE_PREFIX}/")

[20260505T015529Z-bcd908bd] BBox: (-76.6, 3.3, -76.4, 3.55)
[20260505T015529Z-bcd908bd] Periodo: ('2020-01-01', '2024-12-31')
[20260505T015529Z-bcd908bd] Tile: h10v08
[20260505T015529Z-bcd908bd] tmp local: C:\Users\mitgar14\geovision-tmp\maiac\20260505T015529Z-bcd908bd
[20260505T015529Z-bcd908bd] log: C:\Users\mitgar14\geovision-tmp\maiac\20260505T015529Z-bcd908bd\20260505T015529Z-bcd908bd.log
[20260505T015529Z-bcd908bd] destino: abfss://geovision@stanaliticafinal.dfs.core.windows.net/01-bronze/maiac/


In [4]:
# Auth ADLS Gen2: DefaultAzureCredential hereda az login local. Sin Account Keys.
# Auto-crea el filesystem si no existe.
from azure.storage.filedatalake import DataLakeServiceClient

credential = DefaultAzureCredential(exclude_managed_identity_credential=True)
credential.get_token("https://storage.azure.com/.default")  # sanity early failure

fs = adlfs.AzureBlobFileSystem(account_name=STORAGE_ACCOUNT, credential=credential)

dlsc = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential,
)
existing_fs = {f.name for f in dlsc.list_file_systems()}
if FILESYSTEM not in existing_fs:
    dlsc.create_file_system(file_system=FILESYSTEM)
    print(f"[{RUN_ID}] filesystem '{FILESYSTEM}' creado en {STORAGE_ACCOUNT}.")
else:
    print(f"[{RUN_ID}] filesystem '{FILESYSTEM}' ya existe.")

try:
    paths = fs.ls(f"{FILESYSTEM}/")
    print(f"[{RUN_ID}] ADLS OK via DefaultAzureCredential. {len(paths)} entradas en raíz.")
except Exception as exc:
    raise RuntimeError(
        f"No se pudo listar abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/. "
        f"Verificar Storage Blob Data Contributor sobre {STORAGE_ACCOUNT}. Error: {exc}"
    )

[20260505T015529Z-bcd908bd] filesystem 'geovision' ya existe.
[20260505T015529Z-bcd908bd] ADLS OK via DefaultAzureCredential. 2 entradas en raíz.


In [5]:
# earthaccess espera EARTHDATA_USERNAME / EARTHDATA_PASSWORD; el .env del proyecto usa USER/PASS
os.environ["EARTHDATA_USERNAME"] = os.environ["EARTHDATA_USER"]
os.environ["EARTHDATA_PASSWORD"] = os.environ["EARTHDATA_PASS"]

auth = earthaccess.login(strategy="environment", persist=False)
if not auth.authenticated:
    raise RuntimeError(
        "earthaccess no autenticó con EARTHDATA_USERNAME/PASSWORD. "
        "Verificar que la app 'LP DAAC Data Pool' esté autorizada en https://urs.earthdata.nasa.gov/."
    )
print(f"[{RUN_ID}] EDL OK. Token expira: {os.environ.get('EARTHDATA_TOKEN_EXPIRES', 'desconocido')}")

[20260505T015529Z-bcd908bd] EDL OK. Token expira: 2026-06-30


In [6]:
# Búsqueda CMR por bounding_box. CMR no acepta tile MODIS como filtro directo,
# pero h10v08 es el único tile que cubre BBox Cali, así que el bbox actúa como filtro implícito.
results = earthaccess.search_data(
    short_name="MCD19A2",
    version="061",
    bounding_box=BBOX_CALI,
    temporal=PERIODO,
)
print(f"[{RUN_ID}] Granules encontrados: {len(results)}")
if len(results) == 0:
    raise RuntimeError("CMR devolvió 0 granules. Verificar bbox y periodo.")

# Sanity check: el primer granule debe contener 'h10v08' en el nombre
first_name = results[0].data_links()[0].split("/")[-1]
assert TILE_MODIS in first_name, f"Primer granule no es h10v08: {first_name}"
print(f"[{RUN_ID}] Sanity OK. Primer granule: {first_name}")

[20260505T015529Z-bcd908bd] Granules encontrados: 1822
[20260505T015529Z-bcd908bd] Sanity OK. Primer granule: MCD19A2.A2020001.h10v08.061.2023132161306.hdf


c:\Users\mitgar14\Desktop\Ingenieria de Datos e IA - UAO\Semestre 7\Analítica\Semana #12 - #17\Proyecto\.venv\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


In [7]:
def hash_file_dual(path: Path) -> dict:
    """Calcula SHA256 (canónico FAIR) y MD5 (compat. ADLS Content-MD5) en un solo paso."""
    sha = hashlib.sha256()
    md5 = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            sha.update(chunk)
            md5.update(chunk)
    return {"sha256": sha.hexdigest(), "md5": md5.hexdigest(), "size_bytes": path.stat().st_size}


def parse_yyyy_mm(granule_name: str) -> tuple[str, str]:
    """MCD19A2.A2024001.h10v08.061.2024004175601.hdf -> ('2024', '01').

    Fix vs run previo: usar timedelta(days=doy-1) en lugar de datetime(year,1,doy)
    que rompe con DOY > 31 ('day is out of range for month').
    """
    julian = granule_name.split(".")[1]  # 'A2024001'
    year = julian[1:5]
    doy = int(julian[5:8])
    dt = datetime(int(year), 1, 1) + timedelta(days=doy - 1)
    return year, f"{dt.month:02d}"


@contextlib.contextmanager
def redirect_to_log(log_path: Path):
    """Redirige stdout y stderr a `log_path` (append). Útil para silenciar earthaccess/tqdm/rich."""
    with log_path.open("a", encoding="utf-8", buffering=1) as fh:
        with contextlib.redirect_stdout(fh), contextlib.redirect_stderr(fh):
            yield


def log(msg: str) -> None:
    """Log conjunto: stdout + log file (con timestamp)."""
    line = f"[{datetime.now(timezone.utc).isoformat(timespec='seconds')}] {msg}"
    print(line)
    with LOG_FILE.open("a", encoding="utf-8") as fh:
        fh.write(line + "\n")


# Sanity check del fix antes de empezar (DOY 60 -> 1 marzo bisiesto / 1 marzo no)
assert parse_yyyy_mm("MCD19A2.A2020060.h10v08.061.x.hdf") == ("2020", "02"), "fix DOY rota"
assert parse_yyyy_mm("MCD19A2.A2021060.h10v08.061.x.hdf") == ("2021", "03"), "fix DOY rota"
assert parse_yyyy_mm("MCD19A2.A2024366.h10v08.061.x.hdf") == ("2024", "12"), "fix DOY rota"
log("parse_yyyy_mm: sanity OK (DOY 60 bisiesto = feb, no bisiesto = mar; DOY 366 = dic).")

[2026-05-05T01:55:35+00:00] parse_yyyy_mm: sanity OK (DOY 60 bisiesto = feb, no bisiesto = mar; DOY 366 = dic).


In [8]:
# Loop principal: descarga local -> hash -> upload streaming a ADLS -> borrar local.
# Idempotencia: si el remote_path existe, saltar (los 150 granules del run previo
# 20260504T011528Z-75f904dd se saltan automáticamente).
# Output de earthaccess.download (rich + tqdm) se redirige al log file del run.

manifest_entries = []
skipped = []
failed = []

t0 = time.time()
log(f"iniciando loop sobre {len(results)} granules. tmp={TMP_DIR}, log={LOG_FILE}")

for i, granule in enumerate(results, 1):
    name = "<unknown>"
    try:
        urls = granule.data_links()
        if not urls:
            failed.append({"granule": str(granule), "reason": "sin data_links"})
            continue
        url = urls[0]
        name = url.split("/")[-1]
        year, month = parse_yyyy_mm(name)
        remote_path = f"{FILESYSTEM}/{BRONZE_PREFIX}/year={year}/month={month}/{name}"

        # Idempotencia: si ya está en ADLS, saltar (sin tocar LP DAAC).
        if fs.exists(remote_path):
            skipped.append(name)
            if i % 100 == 0:
                log(f"progreso {i}/{len(results)}: ok={len(manifest_entries)} skipped={len(skipped)} failed={len(failed)}")
            continue

        # Descarga (rich/tqdm silenciado al log file).
        with redirect_to_log(LOG_FILE):
            downloaded = earthaccess.download([granule], local_path=str(TMP_DIR), threads=1)
        if not downloaded:
            failed.append({"granule": name, "reason": "earthaccess.download retornó lista vacía"})
            continue
        local = Path(downloaded[0])
        if not local.exists() or local.stat().st_size < 1024:
            failed.append({"granule": name, "reason": f"archivo vacío o ausente: {local}"})
            continue

        # Sanity: header HDF-EOS2 esperado.
        with local.open("rb") as fh:
            header = fh.read(4)
        if header != b"\x0e\x03\x13\x01":
            failed.append({"granule": name, "reason": f"header inválido: {header.hex()}"})
            local.unlink(missing_ok=True)
            continue

        hashes = hash_file_dual(local)

        # Upload streaming a ADLS.
        with local.open("rb") as src, fs.open(remote_path, "wb") as dst:
            while chunk := src.read(8 * 1024 * 1024):
                dst.write(chunk)

        manifest_entries.append({
            "granule": name,
            "remote_path": f"abfss://{remote_path}",
            "year": year,
            "month": month,
            "sha256": hashes["sha256"],
            "md5": hashes["md5"],
            "size_bytes": hashes["size_bytes"],
            "source_url": url,
            "downloaded_at": datetime.now(timezone.utc).isoformat(),
        })

        # Liberar disco local.
        local.unlink(missing_ok=True)

        if i % 50 == 0:
            elapsed = time.time() - t0
            rate = i / max(elapsed, 1)
            eta = (len(results) - i) / max(rate, 1e-9) / 60
            log(f"progreso {i}/{len(results)}: ok={len(manifest_entries)} skipped={len(skipped)} "
                f"failed={len(failed)} | rate={rate:.2f}/s eta={eta:.1f}min")

        # Backoff conservador para LP DAAC (20-60 req/min límite dinámico).
        time.sleep(2)

    except Exception as exc:
        failed.append({"granule": name, "reason": str(exc)[:200]})
        # Backoff exponencial leve en errores (5s, 10s) para no martillar en cascada.
        time.sleep(5 if len(failed) % 5 else 10)

elapsed = time.time() - t0
log(f"loop terminado en {elapsed/60:.1f} min: ok={len(manifest_entries)} "
    f"skipped={len(skipped)} failed={len(failed)}")

[2026-05-05T01:55:35+00:00] iniciando loop sobre 1822 granules. tmp=C:\Users\mitgar14\geovision-tmp\maiac\20260505T015529Z-bcd908bd, log=C:\Users\mitgar14\geovision-tmp\maiac\20260505T015529Z-bcd908bd\20260505T015529Z-bcd908bd.log
[2026-05-05T01:57:49+00:00] progreso 50/1822: ok=11 skipped=39 failed=0 | rate=0.37/s eta=78.9min
[2026-05-05T02:07:42+00:00] progreso 100/1822: ok=61 skipped=39 failed=0 | rate=0.14/s eta=208.7min
[2026-05-05T02:16:46+00:00] progreso 150/1822: ok=111 skipped=39 failed=0 | rate=0.12/s eta=236.0min
[2026-05-05T02:25:35+00:00] progreso 200/1822: ok=161 skipped=39 failed=0 | rate=0.11/s eta=243.3min
[2026-05-05T02:34:25+00:00] progreso 250/1822: ok=211 skipped=39 failed=0 | rate=0.11/s eta=244.2min
[2026-05-05T02:43:18+00:00] progreso 300/1822: ok=261 skipped=39 failed=0 | rate=0.10/s eta=242.1min
[2026-05-05T02:52:16+00:00] progreso 350/1822: ok=311 skipped=39 failed=0 | rate=0.10/s eta=238.4min
[2026-05-05T02:55:46+00:00] progreso 400/1822: ok=330 skipped=70 f

In [9]:
# Manifest YAML (esquema canónico doc 06-manifest-zarr-parquet.md:74-197)
manifest = {
    "dataset": "maiac_aod_mcd19a2_v061",
    "run_id": RUN_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "execution_mode": "local",
    "previous_runs": [
        # los 150 granules de DOY 1-31 vinieron de este run (subidos correctamente,
        # validados size+sha256+header HDF-EOS2 contra manifest 2026-05-04).
        "20260504T011528Z-75f904dd",
    ],
    "source": {
        "provider": "NASA LP DAAC",
        "product": "MCD19A2",
        "version": "061",
        "tile": TILE_MODIS,
        "endpoint": "https://cmr.earthdata.nasa.gov/search",
    },
    "spatial": {
        "bbox_lon_min": BBOX_CALI[0],
        "bbox_lat_min": BBOX_CALI[1],
        "bbox_lon_max": BBOX_CALI[2],
        "bbox_lat_max": BBOX_CALI[3],
    },
    "temporal": {"start": PERIODO[0], "end": PERIODO[1]},
    "counts": {
        "granules_found": len(results),
        "downloaded": len(manifest_entries),
        "skipped_idempotent": len(skipped),
        "failed": len(failed),
    },
    "hash_algo": ["sha256", "md5"],
    "granules": manifest_entries,
    "failures": failed,
}

manifest_path = f"{FILESYSTEM}/{META_PREFIX}/{RUN_ID}.yaml"
with fs.open(manifest_path, "w") as f:
    yaml.safe_dump(manifest, f, allow_unicode=True, sort_keys=False)

total_gb = sum(e["size_bytes"] for e in manifest_entries) / 1e9
log(f"manifest escrito en abfss://{manifest_path}")
log(f"total descargado en este run: {total_gb:.2f} GB ({len(manifest_entries)} granules)")
log(f"saltados por idempotencia: {len(skipped)} granules ya en bucket")

[2026-05-05T06:55:44+00:00] manifest escrito en abfss://geovision/_meta/manifest/maiac/20260505T015529Z-bcd908bd.yaml
[2026-05-05T06:55:44+00:00] total descargado en este run: 11.39 GB (1664 granules)
[2026-05-05T06:55:44+00:00] saltados por idempotencia: 158 granules ya en bucket


In [10]:
# Subir log file a ADLS y limpiar tmp local.
import shutil

remote_log = f"{FILESYSTEM}/{LOG_PREFIX}/{RUN_ID}.log"
with LOG_FILE.open("rb") as src, fs.open(remote_log, "wb") as dst:
    while chunk := src.read(1 * 1024 * 1024):
        dst.write(chunk)
print(f"[{RUN_ID}] log subido a abfss://{remote_log}")

# Cleanup tmp local. Saltar con env MAIAC_KEEP_TMP=1 para auditoría manual.
if os.environ.get("MAIAC_KEEP_TMP") == "1":
    print(f"[{RUN_ID}] MAIAC_KEEP_TMP=1: tmp preservado en {TMP_DIR}")
else:
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    print(f"[{RUN_ID}] tmp limpio. Notebook MAIAC finalizado.")

[20260505T015529Z-bcd908bd] log subido a abfss://geovision/_meta/logs/maiac/20260505T015529Z-bcd908bd.log
[20260505T015529Z-bcd908bd] tmp limpio. Notebook MAIAC finalizado.
